In [ ]:
import requests
import pandas as pd
import os
import re

%pip install requests pandas openpyxl python-dotenv

 

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("CANVAS_BASE_URL"))
print(os.getenv("CANVAS_TOKEN"))

In [ ]:
# 定义API的基本URL和请求头

load_dotenv()

CANVAS_BASE_URL = os.getenv("CANVAS_BASE_URL").rstrip("/")
CANVAS_TOKEN = os.getenv("CANVAS_TOKEN")

API_BASE = f"{CANVAS_BASE_URL}/api/v1"

headers = {
    "Authorization": f"Bearer {CANVAS_TOKEN}",
    "Accept": "application/json"
}

print("Canvas:", CANVAS_BASE_URL)
print("Token loaded:", bool(CANVAS_TOKEN))

Canvas: https://hkust-gz.instructure.com
Token loaded: True


In [ ]:
# 定义获取API需求的函数

def canvas_get(endpoint, params=None):
    """
    Send GET request to Canvas API and return JSON.
    """
    
    url = f"{API_BASE}{endpoint}"
    
    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=30
    )
    
    print(f"GET {response.url}")
    print(f"Status: {response.status_code}")
    
    response.raise_for_status()
    
    return response.json()

In [ ]:
# 测试课程数据

course_id = 3308

course = canvas_get(f"/courses/{course_id}")

course["id"], course["name"]

In [33]:
# 定义2个作业和相应的rubric_id

assignments = {
    23999: {
        "name": "Assignment 1",
        "rubric_id": 1077
    },
    24000: {
        "name": "Assignment 2",
        "rubric_id": 1078
    }
}

In [34]:
# 获取Rubric信息并建立criterion的字典

rubric_cache = {}

for assignment_id, info in assignments.items():
    
    rubric_id = info["rubric_id"]
    
    rubric = canvas_get(
        f"/courses/{course_id}/rubrics/{rubric_id}"
    )
    
    rubric_cache[rubric_id] = rubric

GET https://hkust-gz.instructure.com/api/v1/courses/3308/rubrics/1077
Status: 200
GET https://hkust-gz.instructure.com/api/v1/courses/3308/rubrics/1078
Status: 200


In [ ]:
# 测试输出Rubric的数据字段

rubric_cache[1077].keys()

dict_keys(['id', 'title', 'context_id', 'context_type', 'points_possible', 'reusable', 'public', 'read_only', 'free_form_criterion_comments', 'hide_score_total', 'data', 'rating_order', 'button_display'])

In [ ]:
# 测试Rubric中data字段的信息

rubric_cache[1077]["data"]

[{'description': 'Asking the right question',
  'long_description': '',
  'points': 18.0,
  'id': '1781512456632',
  'criterion_use_range': False,
  'generated': False,
  'ratings': [{'description': 'Exceeds',
    'long_description': '',
    'points': 18.0,
    'criterion_id': '1781512456632',
    'id': '1'},
   {'description': 'Mastery',
    'long_description': '',
    'points': 13.5,
    'criterion_id': '1781512456632',
    'id': '2'},
   {'description': 'Near',
    'long_description': '',
    'points': 9.0,
    'criterion_id': '1781512456632',
    'id': '3'},
   {'description': 'Below',
    'long_description': '',
    'points': 4.5,
    'criterion_id': '1781512456632',
    'id': '4'},
   {'description': 'No Evidence',
    'long_description': '',
    'points': 0.0,
    'criterion_id': '1781512456632',
    'id': '5'}]},
 {'description': 'Logical structure',
  'long_description': '',
  'points': 18.0,
  'id': '1781512487374',
  'criterion_use_range': False,
  'generated': False,
  'rat

In [18]:
# 建立 criterion mapping

def build_criterion_map(rubric):
    """
    Convert Canvas rubric criteria into:
    
    criterion_id -> {
        name,
        description,
        points
    }
    """
    
    criterion_map = {}
    
    for criterion in rubric.get("data", []):
        
        criterion_id = str(criterion["id"])
        
        criterion_map[criterion_id] = {
            "name": criterion.get("description", criterion_id),
            "long_description": criterion.get("long_description"),
            "points": criterion.get("points")
        }
    
    return criterion_map

In [19]:
criterion_maps = {
    rubric_id: build_criterion_map(rubric)
    for rubric_id, rubric in rubric_cache.items()
}

criterion_maps[1077]

{'1781512456632': {'name': 'Asking the right question',
  'long_description': '',
  'points': 18.0},
 '1781512487374': {'name': 'Logical structure',
  'long_description': '',
  'points': 18.0},
 '1781512498763': {'name': 'Accuracy & comprehensive',
  'long_description': '',
  'points': 18.0},
 '1781512512067': {'name': 'Demonstrate critical thinking and propose new ideas',
  'long_description': '',
  'points': 18.0},
 '1781512526265': {'name': 'Formatting, e.g. figures, tables, text, and proper reference',
  'long_description': '',
  'points': 18.0}}

In [23]:
# 把 criterion 名称变成合法的 Excel 列名

def clean_column_name(name):
    """
    Make rubric criterion name suitable for DataFrame/Excel column names.
    """
    
    name = str(name).strip()
    
    # 空格 → _
    name = re.sub(r"\s+", "_", name)
    
    # 删除特殊字符
    name = re.sub(r"[^\w\u4e00-\u9fff-]", "", name)
    
    return name

In [24]:
def build_column_map(criterion_map):
    
    column_map = {}
    
    for criterion_id, info in criterion_map.items():
        
        criterion_name = clean_column_name(info["name"])
        
        column_map[criterion_id] = {
            "score": f"{criterion_name}_score",
            "comment": f"{criterion_name}_comment"
        }
    
    return column_map

In [25]:
column_maps = {
    rubric_id: build_column_map(criterion_map)
    for rubric_id, criterion_map in criterion_maps.items()
}

column_maps[1077]

{'1781512456632': {'score': 'Asking_the_right_question_score',
  'comment': 'Asking_the_right_question_comment'},
 '1781512487374': {'score': 'Logical_structure_score',
  'comment': 'Logical_structure_comment'},
 '1781512498763': {'score': 'Accuracy__comprehensive_score',
  'comment': 'Accuracy__comprehensive_comment'},
 '1781512512067': {'score': 'Demonstrate_critical_thinking_and_propose_new_ideas_score',
  'comment': 'Demonstrate_critical_thinking_and_propose_new_ideas_comment'},
 '1781512526265': {'score': 'Formatting_eg_figures_tables_text_and_proper_reference_score',
  'comment': 'Formatting_eg_figures_tables_text_and_proper_reference_comment'}}

In [35]:
# 获取 Peer Review 原始数据

peer_raw = {}

for assignment_id, info in assignments.items():
    
    rubric_id = info["rubric_id"]
    
    data = canvas_get(
        f"/courses/{course_id}/rubrics/{rubric_id}",
        params={
            "include[]": "peer_assessments",
            "style": "full"
        }
    )
    
    peer_raw[assignment_id] = data

GET https://hkust-gz.instructure.com/api/v1/courses/3308/rubrics/1077?include%5B%5D=peer_assessments&style=full
Status: 200
GET https://hkust-gz.instructure.com/api/v1/courses/3308/rubrics/1078?include%5B%5D=peer_assessments&style=full
Status: 200


In [36]:
peer_raw[23999]["assessments"][:2]

[{'id': 3799,
  'rubric_id': 1077,
  'rubric_association_id': 2051,
  'score': 49.5,
  'artifact_type': 'Submission',
  'artifact_id': 744176,
  'artifact_attempt': 2,
  'assessment_type': 'peer_review',
  'assessor_id': 10716,
  'data': [{'id': '2',
    'points': 13.5,
    'criterion_id': '1781512456632',
    'learning_outcome_id': None,
    'description': 'Mastery',
    'comments_enabled': True,
    'comments': 'Justification for score ：\nThe introduction clearly raises the core research question: “This raises a critical question: Whether enhanced weathering supports an elevated level of net carbon sink efficiency”. The whole paper splits analysis into terrestrial and marine EW, matching the title’s research scope, and the abstract restates this framework consistently.\nHowever, the text never sets clear comparative expectations for land and marine EW at the end of introduction, making the research goal slightly unfocused, so it cannot reach Exceeds standard\n\nConstructive Feedback：

In [37]:
# 获取peer review -> student的对应关系

peer_review_map = {}

for assignment_id in assignments:
    
    peer_reviews = canvas_get(
        f"/courses/{course_id}/assignments/{assignment_id}/peer_reviews"
    )
    
    peer_review_map[assignment_id] = peer_reviews

GET https://hkust-gz.instructure.com/api/v1/courses/3308/assignments/23999/peer_reviews
Status: 200
GET https://hkust-gz.instructure.com/api/v1/courses/3308/assignments/24000/peer_reviews
Status: 200


In [ ]:
# 测试输出peer_review_map的前2条记录

peer_review_map[23999][:2]

[{'id': 7009,
  'user_id': 393,
  'assessor_id': 10366,
  'asset_id': 742934,
  'asset_type': 'Submission',
  'workflow_state': 'completed'},
 {'id': 6959,
  'user_id': 393,
  'assessor_id': 10641,
  'asset_id': 742934,
  'asset_type': 'Submission',
  'workflow_state': 'completed'}]

In [ ]:
# 建立 artifact -> student 的对应关系

artifact_to_student = {}

for assignment_id, peer_reviews in peer_review_map.items():
    
    artifact_to_student[assignment_id] = {}
    
    for review in peer_reviews:
        
        artifact_id = review.get("asset_id")
        student_id = review.get("user_id")
        
        artifact_to_student[assignment_id][artifact_id] = student_id

In [42]:
# 测试

artifact_to_student[23999]

{742934: 393,
 742060: 6399,
 743499: 6470,
 742954: 6471,
 742061: 6500,
 742955: 6501,
 742230: 10352,
 744161: 10366,
 742232: 10380,
 743500: 10395,
 743502: 10417,
 742233: 10423,
 744162: 10439,
 743503: 10441,
 744163: 10466,
 742236: 10467,
 743504: 10473,
 742908: 10476,
 742238: 10559,
 742240: 10571,
 742241: 10574,
 742242: 10584,
 742244: 10586,
 744556: 10625,
 742245: 10641,
 746183: 10646,
 745659: 10674,
 744165: 10687,
 742546: 10716,
 742248: 10722,
 743505: 10730,
 744166: 10774,
 743506: 10779,
 742251: 10780,
 743850: 10781,
 744317: 10790,
 741888: 14395,
 744176: 14511,
 743993: 14513}

In [43]:
# 把 Peer Review 转成最终表格的一行一行

peer_rows = []

for assignment_id, info in assignments.items():
    
    rubric_id = info["rubric_id"]
    
    rubric = peer_raw[assignment_id]
    
    criterion_map = criterion_maps[rubric_id]
    column_map = column_maps[rubric_id]
    
    assessments = rubric.get("assessments", [])
    
    for assessment in assessments:
        
        # 只保留 peer review
        if assessment.get("assessment_type") != "peer_review":
            continue
        
        artifact_id = assessment.get("artifact_id")
        assessor_id = assessment.get("assessor_id")
        
        # artifact → student
        student_id = artifact_to_student[assignment_id].get(
            artifact_id
        )
        
        # 先建立这一位 reviewer 的一整行
        row = {
            "student_id": student_id,
            "assignment_id": assignment_id,
            "assessment_type": "peer_review",
            "assessor_id": assessor_id,
            "assessment_id": assessment.get("id"),
            "artifact_id": artifact_id,
            "total_score": assessment.get("score")
        }
        
        # 再把每个 criterion 放进去
        for criterion in assessment.get("data", []):
            
            criterion_id = str(
                criterion.get("criterion_id")
            )
            
            if criterion_id not in column_map:
                continue
            
            score_column = column_map[criterion_id]["score"]
            comment_column = column_map[criterion_id]["comment"]
            
            row[score_column] = criterion.get("points")
            row[comment_column] = criterion.get("comments")
        
        peer_rows.append(row)

KeyError: 1078

In [ ]:
# 测试

peer_df = pd.DataFrame(peer_rows)

In [45]:
# 查看

peer_df.head()

,student_id,assignment_id,assessment_type,assessor_id,assessment_id,artifact_id,total_score,Asking_the_right_question_score,Asking_the_right_question_comment,Logical_structure_score,Logical_structure_comment,Accuracy__comprehensive_score,Accuracy__comprehensive_comment,Demonstrate_critical_thinking_and_propose_new_ideas_score,Demonstrate_critical_thinking_and_propose_new_ideas_comment,Formatting_eg_figures_tables_text_and_proper_reference_score,Formatting_eg_figures_tables_text_and_proper_reference_comment
0,14511,23999,peer_review,10716,3799,744176,49.5,13.5,Justification for score ：\nThe introduction cl...,9.0,Justification for score：\nThe basic structure ...,9.0,Justification：\nCore data and chemical equatio...,4.5,Justification：\nAll critical viewpoints only r...,13.5,Justification：\nAll figures clearly mark sourc...
1,6399,23999,peer_review,10716,3800,742060,49.5,13.5,Justification：\nThe introduction clearly state...,13.5,Justification\nThe paper adopts a tight linear...,9.0,"Justification\nMost chemical mechanisms, numer...",9.0,Justification\nThe paper shows basic independe...,4.5,Justification\nSevere formatting defects drag ...
2,6399,23999,peer_review,10779,3812,742060,80.0,16.5,The report presents a clear and course-relevan...,16.0,The report has a logical progression from body...,16.0,The report gives a detailed and generally accu...,15.5,The strongest critical-thinking section is Sec...,16.0,"The report uses numbered headings, page number..."
3,10730,23999,peer_review,10584,3831,743505,40.5,9.0,The topic is relevant to everyday chemistry an...,9.0,The report follows a generally understandable ...,9.0,The report covers a useful range of shampoo in...,9.0,The source-reliability section shows an effort...,4.5,The formatting and referencing require substan...
4,10423,23999,peer_review,10779,3818,742233,74.0,16.5,The report asks a clear and important question...,16.0,The structure is easy to follow. It moves logi...,15.0,The report includes useful quantitative eviden...,14.5,The report demonstrates strong critical thinki...,12.0,"The report includes a clear title, five keywor..."


In [46]:
# 处理 TA Grading

submission_raw = {}

for assignment_id in assignments:
    
    submissions = canvas_get(
        f"/courses/{course_id}/assignments/{assignment_id}/submissions",
        params={
            "per_page": 100
        }
    )
    
    submission_raw[assignment_id] = submissions

GET https://hkust-gz.instructure.com/api/v1/courses/3308/assignments/23999/submissions?per_page=100
Status: 200
GET https://hkust-gz.instructure.com/api/v1/courses/3308/assignments/24000/submissions?per_page=100
Status: 200


In [47]:
# 解析TA grading

ta_rows = []

for assignment_id, submissions in submission_raw.items():
    
    rubric_id = assignments[assignment_id]["rubric_id"]
    
    criterion_map = criterion_maps[rubric_id]
    column_map = column_maps[rubric_id]
    
    for submission in submissions:
        
        student_id = submission.get("user_id")
        
        rubric_assessment = submission.get(
            "rubric_assessment"
        )
        
        # 如果这里没有 rubric assessment，
        # 说明这个学生没有 rubric grading
        if not rubric_assessment:
            continue
        
        row = {
            "student_id": student_id,
            "assignment_id": assignment_id,
            "assessment_type": "grading",
            "assessor_id": submission.get("grader_id"),
            "assessment_id": None,
            "artifact_id": submission.get("id"),
            "total_score": None
        }
        
        total_score = 0
        
        for criterion_id, criterion_data in rubric_assessment.items():
            
            criterion_id = str(criterion_id)
            
            if criterion_id not in column_map:
                continue
            
            points = criterion_data.get("points")
            
            score_column = column_map[criterion_id]["score"]
            comment_column = column_map[criterion_id]["comment"]
            
            row[score_column] = points
            row[comment_column] = criterion_data.get(
                "comments"
            )
            
            # 用 criterion points 求 rubric 总分
            if points is not None:
                total_score += points
        
        row["total_score"] = total_score
        
        ta_rows.append(row)

KeyError: 1078

In [48]:
# 转化成dataframe

ta_df = pd.DataFrame(ta_rows)

In [50]:
# 查看

ta_df.head()

""
